In [1]:
from transformers import pipeline
import os
import gradio as gr

/home/rakesh/Downloads/ShortTok/2.NoiseCancellation/NoiseCancellation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
asr = pipeline(task="automatic-speech-recognition",
               model="distil-whisper/distil-small.en",
               model_kwargs={"cache_dir": "models"})

print("asr_freq:", asr.feature_extractor.sampling_rate)

Loading weights: 100%|██████████| 287/287 [00:00<00:00, 13817.01it/s]


asr_freq: 16000


In [ ]:
def transcribe_speech(filepath):
    if filepath is None:
        gr.Warning("No audio found, please retry.")
        return ""
    output = asr(filepath)
    return output["text"]

mic_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="microphone",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
)

file_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="upload",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
)

demo = gr.Blocks()

with demo:
    gr.TabbedInterface(
        [mic_transcribe,
         file_transcribe],
        ["Transcribe Microphone",
         "Transcribe Audio File"],
    )

demo.launch(share=True, 
            server_port=int(os.environ.get('PORT1', 8000)))

* Running on local URL:  http://127.0.0.1:8000
* Running on public URL: https://0f82ea595deca030ad.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [5]:
import sys
import types
import numpy as np
import librosa
import torch
import torchaudio
from pydub import AudioSegment

sys.modules["torchaudio.backend"] = types.ModuleType("torchaudio.backend")
sys.modules["torchaudio.backend.common"] = types.ModuleType("torchaudio.backend.common")
sys.modules["torchaudio.backend.common"].AudioMetaData = object
from df.enhance import enhance, init_df

CLIPS = {
    "Bird chirp": "noise/bird.wav",
    "Train": "noise/train.wav",
    "Traffic": "noise/traffic.wav",
    "Factory machine": "noise/machine.wav",
}

def wave(samples):
    waveform = np.asarray(samples, np.float32)
    waveform = librosa.to_mono(waveform.T if waveform.ndim > 1 else waveform)
    return waveform / 32768 if waveform.size and np.max(np.abs(waveform)) > 1.5 else waveform

def transcribe(audio):
    if audio is None:
        return ""
    sr, samples = audio
    rate = asr.feature_extractor.sampling_rate
    waveform = librosa.resample(wave(samples), orig_sr=sr, target_sr=rate)
    return asr({"array": waveform, "sampling_rate": rate})["text"]

def audio_segment(samples, sr):
    pcm = (np.clip(samples, -1, 1) * 32767).astype(np.int16)
    return AudioSegment(pcm.tobytes(), frame_rate=int(sr), sample_width=2, channels=1)

def mix(audio, snr_db, *flags):
    if audio is None:
        return None
    sr, samples = audio
    speech = wave(samples)
    if speech.size == 0:
        return None
    mixed = audio_segment(speech, sr)
    speech_db = mixed.dBFS
    for noise_path, selected in zip(CLIPS.values(), flags):
        if not selected:
            continue
        noise, _ = librosa.load(noise_path, sr=sr, mono=True)
        noise = np.tile(noise, int(np.ceil(len(speech) / len(noise))))[:len(speech)]
        clip = audio_segment(noise, sr)
        gain_db = speech_db - clip.dBFS - float(snr_db)
        mixed = mixed.overlay(clip.apply_gain(gain_db))
    return mixed.frame_rate, np.array(mixed.get_array_of_samples(), dtype=np.int16)

df_model, df_state, _ = init_df(log_file=None)

def denoise(audio):
    if audio is None:
        return None
    sr, samples = audio
    resampled = librosa.resample(wave(samples), orig_sr=sr, target_sr=df_state.sr())
    denoised = enhance(df_model, df_state, torch.from_numpy(np.ascontiguousarray(resampled)).unsqueeze(0)).squeeze(0).numpy()
    return df_state.sr(), (np.clip(denoised, -1, 1) * 32767).astype(np.int16)

def refresh(audio, snr_db, *flags):
    noisy = mix(audio, snr_db, *flags)
    denoised = denoise(noisy)
    return noisy, transcribe(noisy), denoised, transcribe(denoised)

def on_audio(audio, snr_db, *flags):
    noisy, noisy_text, denoised, denoised_text = refresh(audio, snr_db, *flags)
    return transcribe(audio), noisy, noisy_text, denoised, denoised_text

def panel(source):
    audio = gr.Audio(sources=source, type="numpy", label="Clean speech")
    clear = gr.Button("Clear")
    transcript = gr.Textbox(label="Transcription", lines=3)
    with gr.Row():
        boxes = [gr.Checkbox(True, label=label) for label in CLIPS]
    snr = gr.Slider(0, 20, value=5, step=1, label="SNR in dB (lower = louder noise)")
    noisy = gr.Audio(label="Noisy audio")
    with gr.Row():
        with gr.Column():
            noisy_text = gr.Textbox(label="Noisy transcription", lines=3)
        with gr.Column():
            denoised = gr.Audio(label="Denoised audio")
            denoised_text = gr.Textbox(label="Whisper transcription", lines=3)
    inputs = [audio, snr, *boxes]
    clear.click(lambda: (None, "", 5, None, "", None, ""), outputs=[audio, transcript, snr, noisy, noisy_text, denoised, denoised_text])
    event = audio.stop_recording if source == "microphone" else audio.upload
    event(on_audio, inputs, [transcript, noisy, noisy_text, denoised, denoised_text])
    snr.input(refresh, inputs, [noisy, noisy_text, denoised, denoised_text], show_progress="hidden", trigger_mode="always_last")
    for box in boxes:
        box.change(refresh, inputs, [noisy, noisy_text, denoised, denoised_text], show_progress="hidden")

with gr.Blocks() as microphone:
    panel("microphone")
with gr.Blocks() as upload:
    panel("upload")

noisy_demo = gr.Blocks()
with noisy_demo:
    gr.TabbedInterface([microphone, upload], ["Microphone", "Upload Audio File"])

noisy_demo.launch(server_port=int(os.environ.get("PORT2", 8001)))

2026-10-07 13:36:22 | INFO     | DF | Running on torch 2.14.1+cu130
2026-10-07 13:36:22 | INFO     | DF | Running on host rakesh-Katana-A15-AI-B8VF
2026-10-07 13:36:22 | INFO     | DF | Git commit: 0cbde07, branch: main
2026-10-07 13:36:22 | INFO     | DF | Loading model settings of DeepFilterNet3
2026-10-07 13:36:22 | INFO     | DF | Using DeepFilterNet3 model at /home/rakesh/.cache/DeepFilterNet/DeepFilterNet3
2026-10-07 13:36:22 | INFO     | DF | Initializing model `deepfilternet3`
2026-10-07 13:36:22 | INFO     | DF | Found checkpoint /home/rakesh/.cache/DeepFilterNet/DeepFilterNet3/checkpoints/model_120.ckpt.best with epoch 120
2026-10-07 13:36:22 | INFO     | DF | Running on device cuda:0
2026-10-07 13:36:22 | INFO     | DF | Model loaded
* Running on local URL:  http://127.0.0.1:8001
* To create a public link, set `share=True` in `launch()`.


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
